In [1]:
import os
import shutil
import re

# Folder setup
log_folder = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs"
run_log_path = os.path.join(log_folder, "run_log.txt")
success_folder = os.path.join(log_folder, "successful")
fail_folder = os.path.join(log_folder, "failed")

os.makedirs(success_folder, exist_ok=True)
os.makedirs(fail_folder, exist_ok=True)

# Parse run_log.txt
success_hashes = []
fail_hashes = []

with open(run_log_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    match = re.match(r"==== Processing (.+\.mod) ====", line)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)

# Move .log files based on success/failure
def move_log_files(hashes, destination):
    for hash_id in hashes:
        log_file = f"{hash_id}.log"
        src = os.path.join(log_folder, log_file)
        dst = os.path.join(destination, log_file)
        if os.path.exists(src):
            shutil.copy2(src, dst)

move_log_files(success_hashes, success_folder)
move_log_files(fail_hashes, fail_folder)

print(f"✅ Moved {len(success_hashes)} successful logs to {success_folder}")
print(f"❌ Moved {len(fail_hashes)} failed logs to {fail_folder}")


✅ Moved 426 successful logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/successful
❌ Moved 128 failed logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/failed


In [3]:
log_files = set(f.replace(".log", "") for f in os.listdir(log_folder) if f.endswith(".log"))
mentioned_hashes = set(success_hashes + fail_hashes)
unmatched_logs = log_files - mentioned_hashes
print("Not matched in run_log.txt:", unmatched_logs)

Not matched in run_log.txt: set()


In [4]:
# Check for mod files mentioned in run_log.txt but not labeled as success or failure
processed_hashes = set()
for i, line in enumerate(lines):
    match = re.match(r"==== Processing (.+\.mod) ====", line)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        processed_hashes.add(hash_id)

        # Check next line for status
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)

# Now find any processed hashes that were not categorized
uncategorized = processed_hashes - set(success_hashes) - set(fail_hashes)
print("✅ Total processed:", len(processed_hashes))
print("✅ Total successes:", len(success_hashes))
print("❌ Total failures:", len(fail_hashes))
print("🤷 Not classified:", len(uncategorized))
print("List of not classified:", uncategorized)


✅ Total processed: 554
✅ Total successes: 852
❌ Total failures: 256
🤷 Not classified: 0
List of not classified: set()


In [5]:
for h in success_hashes:
    if h not in processed_hashes:
        print("Unexpected success hash not in processed set:", h)


In [6]:
success_hashes = []
fail_hashes = []
processed_hashes = []

for i in range(len(lines) - 1):
    current = lines[i]
    next_line = lines[i + 1]

    match = re.match(r"==== Processing (.+\.mod) ====", current)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        processed_hashes.append(hash_id)

        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)


In [9]:
import os
import shutil

# Get absolute path to the project root (go up one level from 'code/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Absolute paths
failed_log_dir = os.path.join(project_root, "logs", "failed")
mod_src_dir = os.path.join(project_root, "data", "raw", "nmodl")
mod_dst_dir = os.path.join(project_root, "data", "raw", "nmodl-failed")

# Create the destination folder if it doesn't exist
os.makedirs(mod_dst_dir, exist_ok=True)

# Collect .log files in the failed folder
failed_hashes = [f.replace(".log", "") for f in os.listdir(failed_log_dir) if f.endswith(".log")]

# Copy matching .mod files
copied = 0
for hash_id in failed_hashes:
    mod_filename = f"{hash_id}.mod"
    src = os.path.join(mod_src_dir, mod_filename)
    dst = os.path.join(mod_dst_dir, mod_filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        print(f"⚠️ Missing .mod file for: {mod_filename}")

print(f"✅ Copied {copied} .mod files to '{mod_dst_dir}'")


✅ Copied 128 .mod files to '/gpfs/gibbs/project/mcdougal/imc33/mod-extract/data/raw/nmodl-failed'


In [7]:
!git add .
!git commit -m "checking log files"
!git push

[main 17fa502] checking log files
 2 files changed, 208 insertions(+), 119 deletions(-)
 create mode 100644 code/01d_split_logs.ipynb
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 64 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 98.47 KiB | 3.79 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:innacohen/mod-extract.git
   5de1c25..17fa502  main -> main
